In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pystac_client
import planetary_computer as pc
import rasterio
from rasterio.warp import transform as rio_transform
from tqdm import tqdm
import os
import time

tqdm.pandas()

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

PC_CATALOG = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace
)

GSW_ASSETS = ["change", "extent", "occurrence", "recurrence", "seasonality", "transitions"]

def _sample_asset(asset_href, lon, lat):
    with rasterio.open(asset_href) as src:

        if src.crs is None:
            return np.nan

        # Convert coordinates if needed
        if src.crs.to_string() != "EPSG:4326":
            xs, ys = rio_transform("EPSG:4326", src.crs, [lon], [lat])
            x, y = xs[0], ys[0]
        else:
            x, y = lon, lat

        val = next(src.sample([(x, y)]))[0]

        if src.nodata is not None and val == src.nodata:
            return np.nan

        return float(val)

def compute_gsw_features(row, max_retries=5):
    lat = float(row["Latitude"])
    lon = float(row["Longitude"])

    point = {"type": "Point", "coordinates": [lon, lat]}

    for attempt in range(max_retries):
        try:
            search = PC_CATALOG.search(
                collections=["jrc-gsw"],
                intersects=point,
                limit=5
            )

            items = list(search.items())

            if len(items) == 0:
                return {f"gsw_{k}": np.nan for k in GSW_ASSETS}

            item = items[0]

            out = {}
            for k in GSW_ASSETS:
                if k not in item.assets:
                    out[f"gsw_{k}"] = np.nan
                    continue

                href = item.assets[k].href
                out[f"gsw_{k}"] = _sample_asset(href, lon, lat)

            return out

        except Exception:
            time.sleep(2 ** attempt)

    return {f"gsw_{k}": np.nan for k in GSW_ASSETS}


In [2]:
train_df = pd.read_csv(
    os.path.join(PROJECT_ROOT + "/Provided Datasets", "water_quality_training_dataset.csv")
)

print(f"Training dataset shape: {train_df.shape}")
train_df.head()


Training dataset shape: (9319, 6)


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0


In [3]:
chunk_size = 200
output_path_train = "../New Datasets/jrc_gsw_features_training.csv"

def count_rows_in_csv(path: str) -> int:
    with open(path, "r", encoding="utf-8") as f:
        return max(sum(1 for _ in f) - 1, 0)

start_idx = 0
expected_cols = None

if os.path.exists(output_path_train):
    header_cols = pd.read_csv(output_path_train, nrows=0).columns.tolist()
    expected_cols = header_cols
    start_idx = count_rows_in_csv(output_path_train)

print("Running JRC GSW feature extraction for TRAINING data (chunked)...")
print(f"Total rows in training dataset: {len(train_df)}")
print(f"Output file: {output_path_train}")
print(f"Chunk size: {chunk_size}")
print(f"Resuming from row index: {start_idx}")

for chunk_start in range(start_idx, len(train_df), chunk_size):
    chunk_end = min(chunk_start + chunk_size, len(train_df))
    chunk = train_df.iloc[chunk_start:chunk_end].copy()

    # Compute features row-by-row
    feats = chunk.progress_apply(compute_gsw_features, axis=1)
    feats_df = pd.DataFrame(list(feats))

    # Join features onto the original chunk
    out_chunk = pd.concat([chunk.reset_index(drop=True), feats_df.reset_index(drop=True)], axis=1)

    # Enforce stable column order if resuming
    if expected_cols is None:
        expected_cols = out_chunk.columns.tolist()
    else:
        for col in expected_cols:
            if col not in out_chunk.columns:
                out_chunk[col] = np.nan
        out_chunk = out_chunk[expected_cols]

    # Append to CSV (write header only if file doesn't exist yet)
    write_header = not os.path.exists(output_path_train)
    out_chunk.to_csv(output_path_train, mode="a", header=write_header, index=False)

    print(f"Wrote rows {chunk_start} to {chunk_end-1}")


Running JRC GSW feature extraction for TRAINING data (chunked)...
Total rows in training dataset: 9319
Output file: ../New Datasets/jrc_gsw_features_training.csv
Chunk size: 200
Resuming from row index: 0


 32%|███▏      | 63/200 [01:05<02:22,  1.04s/it]


KeyboardInterrupt: 

In [3]:
val_df = pd.read_csv("../submission_template.csv")
print(f"Validation dataset shape: {val_df.shape}")
val_df.head()


Validation dataset shape: (200, 6)


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,NaN,NaN,NaN
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,NaN,NaN,NaN
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,NaN,NaN,NaN


In [4]:
chunk_size = 200
output_path_val = "../New Datasets/jrc_gsw_features_validation.csv"

start_idx = 0
expected_cols_val = None

if os.path.exists(output_path_val):
    header_cols_val = pd.read_csv(output_path_val, nrows=0).columns.tolist()
    expected_cols_val = header_cols_val
    start_idx = count_rows_in_csv(output_path_val)

print("Running JRC GSW feature extraction for VALIDATION data (chunked)...")
print(f"Total rows in validation dataset: {len(val_df)}")
print(f"Output file: {output_path_val}")
print(f"Chunk size: {chunk_size}")
print(f"Resuming from row index: {start_idx}")

for chunk_start in range(start_idx, len(val_df), chunk_size):
    chunk_end = min(chunk_start + chunk_size, len(val_df))
    chunk = val_df.iloc[chunk_start:chunk_end].copy()

    feats = chunk.progress_apply(compute_gsw_features, axis=1)
    feats_df = pd.DataFrame(list(feats))

    out_chunk = pd.concat([chunk.reset_index(drop=True), feats_df.reset_index(drop=True)], axis=1)

    if expected_cols_val is None:
        expected_cols_val = out_chunk.columns.tolist()
    else:
        for col in expected_cols_val:
            if col not in out_chunk.columns:
                out_chunk[col] = np.nan
        out_chunk = out_chunk[expected_cols_val]

    write_header = not os.path.exists(output_path_val)
    out_chunk.to_csv(output_path_val, mode="a", header=write_header, index=False)

    print(f"Wrote rows {chunk_start} to {chunk_end-1}")


Running JRC GSW feature extraction for VALIDATION data (chunked)...
Total rows in validation dataset: 200
Output file: ../New Datasets/jrc_gsw_features_validation.csv
Chunk size: 200
Resuming from row index: 0


100%|██████████| 200/200 [01:37<00:00,  2.05it/s]

Wrote rows 0 to 199
